# Linear Regression Model Comparison

This notebook compares two linear regression models using the Boston Housing dataset.

## Import libraries

Import the tools needed to load the data, build the models, and compare the results.

In [1]:
# Import the libraries used in the notebook.
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

## Load the data

Load the Boston Housing data and look at the first five rows.

In [2]:
# load_boston() was removed from scikit-learn, so load the same data from a CSV instead.
data_url = "https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv"
housing = pd.read_csv(data_url)
housing.columns = housing.columns.str.upper()

print(housing.shape)
housing.head()

(506, 14)


,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2


## Prepare the data

Separate the target from the features and use the same train/test split for both models.

In [3]:
# MEDV is the value that both models will predict.
X = housing.drop(columns="MEDV")
y = housing["MEDV"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

## Model 1: remove PTRATIO

Build the first model with every feature except `PTRATIO`.

In [4]:
# Remove PTRATIO, then train the first model.
X_train_no_ptratio = X_train.drop(columns="PTRATIO")
X_test_no_ptratio = X_test.drop(columns="PTRATIO")

model_no_ptratio = LinearRegression()
model_no_ptratio.fit(X_train_no_ptratio, y_train)
predictions_no_ptratio = model_no_ptratio.predict(X_test_no_ptratio)

## Model 2: remove B

Build the second model with every feature except `B`.

In [5]:
# Remove B, then train the second model.
X_train_no_b = X_train.drop(columns="B")
X_test_no_b = X_test.drop(columns="B")

model_no_b = LinearRegression()
model_no_b.fit(X_train_no_b, y_train)
predictions_no_b = model_no_b.predict(X_test_no_b)

## Compare the models

Use R-squared, RMSE, and MAE to compare both models on the test data.

In [6]:
# Calculate the same three scores for each model.
def get_scores(actual, predicted):
    return {
        "R-squared": r2_score(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "MAE": mean_absolute_error(actual, predicted),
    }


comparison = pd.DataFrame(
    {
        "Without PTRATIO": get_scores(y_test, predictions_no_ptratio),
        "Without B": get_scores(y_test, predictions_no_b),
    }
).T

comparison

,R-squared,RMSE,MAE
Without PTRATIO,0.629049,5.215673,3.530902
Without B,0.689397,4.772600,3.111377


## Comments

- I removed `PTRATIO` because some good neighborhoods have public schools with higher pupil-teacher ratios, so I did not want that feature to affect this version of the model.
- I removed `B` from the second model because that feature is based on racial demographics, so I did not want to use it in that version.
- The model without `B` performed better. Its R-squared was about 0.689, compared with about 0.629 for the model without `PTRATIO`. It also had lower RMSE and MAE values. For this train/test split, removing `B` hurt the model less than removing `PTRATIO`.